# Run one seed of a sweep on Colab

1. Upload `influence-pruning.zip` to the top level of your Google Drive (**My Drive**).
2. **Runtime → Change runtime type → GPU**.
3. Set `SWEEP` and `SEEDS` below. Give every machine a different seed. For example: your PC runs `0`, Colab runs `1`, a second Colab or Kaggle account runs `2`.
4. **Runtime → Run all**.

All outputs (scores, subsets, results) go to Google Drive, so a disconnect loses at most the run in progress. After a disconnect, reconnect and press **Run all** again; finished runs are skipped.
On the free tier, keep this tab open: closing the browser can end the session.

When the sweep is done, copy `My Drive/influence-pruning-runs/<run name>/results/*.json` into the same `runs/<run name>/results/` folder on your PC. File names include the seed, so nothing collides. Then run `python -m dpif.aggregate runs/<run name>`.

In [ ]:
SWEEP = "sweeps/tier2_cifar10_main.yaml"   # which sweep
SEEDS = "1"                                  # this machine's seed(s), e.g. "1" or "1,2"
ZIP   = "/content/drive/MyDrive/influence-pruning.zip"
RUNS  = "/content/drive/MyDrive/influence-pruning-runs"   # persistent outputs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
!rm -rf /content/influence-pruning && unzip -oq "$ZIP" -d /content
os.chdir('/content/influence-pruning')
!pip install -q pyyaml pandas tabulate matplotlib pytest
os.makedirs(RUNS, exist_ok=True)
!rm -rf runs && ln -s "$RUNS" runs
!python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"
!python -m pytest -q tests 2>&1 | tail -1

In [ ]:
# Long-running cell. Re-run it after a disconnect: completed runs are skipped.
!python -m dpif.sweep "$SWEEP" --seeds "$SEEDS" --device cuda --max-hours 11 2>&1 | tee -a "runs/colab_seed{SEEDS}.log" | grep -v "^  epoch" 

In [ ]:
# Optional: table of whatever has finished so far (all seeds present in this Drive folder)
import yaml
name = yaml.safe_load(open(SWEEP)).get('name') or yaml.safe_load(open(yaml.safe_load(open(SWEEP))['config']))['name']
!python -m dpif.aggregate "runs/{name}" 